# Combined Data Model Training and Comparison

## Setup

In [14]:
from pathlib import Path
import os
import warnings

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_curve,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
SEQUENCE_LENGTH = 4

## Load the datasets

In [15]:
DATA_DIR = Path(".")
if not (DATA_DIR / "discount_price_final_dataset.csv").exists():
    DATA_DIR = Path("ML/Price-Prediction/price_prediction_by_shivam")

files = {
    "2025 only": DATA_DIR / "discount_price_final_dataset.csv",
    "2024 + 2025 VIC": DATA_DIR / "discount_price_combined_2024_2025_vic.csv",
}
missing = [str(file) for file in files.values() if not file.exists()]
if missing:
    raise FileNotFoundError("Run the preprocessing notebooks first. Missing: " + ", ".join(missing))

datasets = {
    name: pd.read_csv(file, parse_dates=["catalogue_start_date", "target_week"])
    for name, file in files.items()
}

for name, frame in datasets.items():
    print(
        f"{name}: {len(frame):,} rows, {frame['product_id'].nunique():,} products, "
        f"{frame['catalogue_start_date'].min().date()} to {frame['catalogue_start_date'].max().date()}"
    )

2025 only: 45,431 rows, 6,784 products, 2025-01-01 to 2025-12-31
2024 + 2025 VIC: 59,539 rows, 8,508 products, 2024-09-11 to 2025-12-31


## Chronological train, validation and test split

In [16]:
dates_2025 = sorted(
    datasets["2025 only"].loc[
        datasets["2025 only"]["classification_eligible"].eq(1),
        "catalogue_start_date",
    ].unique()
)
validation_start = pd.Timestamp(dates_2025[-16])
test_start = pd.Timestamp(dates_2025[-8])

def split_masks(frame):
    dates = frame["catalogue_start_date"]
    train = dates.lt(validation_start)
    validation = dates.ge(validation_start) & dates.lt(test_start)
    test = dates.ge(test_start)
    return train, validation, test

print(f"Training ends before: {validation_start.date()}")
print(f"Validation: {validation_start.date()} to {(test_start - pd.Timedelta(days=1)).date()}")
print(f"Test starts: {test_start.date()}")

for name, frame in datasets.items():
    eligible = frame[frame["classification_eligible"].eq(1)]
    train, validation, test = split_masks(eligible)
    print(f"{name}: train={train.sum():,}, validation={validation.sum():,}, test={test.sum():,}")

Training ends before: 2025-09-09
Validation: 2025-09-09 to 2025-11-04
Test starts: 2025-11-05
2025 only: train=30,511, validation=7,435, test=6,694
2024 + 2025 VIC: train=44,619, validation=7,435, test=6,694


## Shared features and metrics

In [17]:
CLASSIFICATION_CATEGORICAL = ["season", "promo_type", "regular_price_source"]
CLASSIFICATION_FEATURES = [
    "catalogue_observed", "effective_price", "price_inferred", "price_trustworthy",
    "price_outlier", "is_special", "discount_percent", "history_count",
    "cold_start_product", "price_lag_1", "price_lag_2", "price_lag_4",
    "discount_lag_1", "avg_price_4w", "avg_price_8w",
    "special_frequency_4w", "special_frequency_8w", "weeks_since_last_special",
    "weeks_since_regular_price", "week_of_year", "month", "quarter",
    "season", "promo_type", "regular_price_source",
]

REGRESSION_CATEGORICAL = ["retailer", "region", "season", "promo_type", "regular_price_source"]
REGRESSION_FEATURES = [
    "retailer", "region", "catalogue_observed", "special_price_clean",
    "regular_price_filled", "effective_price", "price_inferred",
    "price_trustworthy", "price_outlier", "is_special", "discount_percent",
    "discount_percent_trustworthy", "history_count", "cold_start_product",
    "price_lag_1", "price_lag_2", "price_lag_4", "discount_lag_1",
    "avg_price_4w", "avg_price_8w", "special_frequency_4w",
    "special_frequency_8w", "weeks_since_last_special",
    "weeks_since_regular_price", "week_of_year", "month", "quarter",
    "season", "promo_type", "regular_price_source",
]

LSTM_FEATURES = [
    "catalogue_observed", "effective_price", "price_inferred", "price_trustworthy",
    "price_outlier", "is_special", "discount_percent", "history_count",
    "cold_start_product", "price_lag_1", "price_lag_2", "price_lag_4",
    "discount_lag_1", "avg_price_4w", "avg_price_8w",
    "special_frequency_4w", "special_frequency_8w", "weeks_since_last_special",
    "weeks_since_regular_price", "week_of_year", "month", "quarter",
]

def best_f1_threshold(y_true, probability):
    precision, recall, thresholds = precision_recall_curve(y_true, probability)
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    return float(thresholds[np.argmax(f1)])

def classification_scores(y_true, probability, threshold):
    prediction = (probability >= threshold).astype(int)
    return {
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, prediction),
        "Balanced Accuracy": balanced_accuracy_score(y_true, prediction),
        "Precision": precision_score(y_true, prediction, zero_division=0),
        "Recall": recall_score(y_true, prediction, zero_division=0),
        "F1": f1_score(y_true, prediction, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, probability),
        "PR-AUC": average_precision_score(y_true, probability),
    }

## Initialize the existing classification models

In [18]:
def make_classifier_models(y_train):
    positive_weight = (len(y_train) - y_train.sum()) / y_train.sum()
    return {
        "Logistic Regression": make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE),
        ),
        "Random Forest": make_pipeline(
            SimpleImputer(strategy="median"),
            RandomForestClassifier(
                n_estimators=200, max_depth=14, min_samples_leaf=2,
                class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE,
            ),
        ),
        "HistGradientBoosting": make_pipeline(
            SimpleImputer(strategy="median"),
            HistGradientBoostingClassifier(
                max_iter=200, learning_rate=0.07, max_leaf_nodes=31,
                class_weight="balanced", random_state=RANDOM_STATE,
            ),
        ),
        "XGBoost": make_pipeline(
            SimpleImputer(strategy="median"),
            XGBClassifier(
                n_estimators=250, max_depth=5, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.8,
                scale_pos_weight=positive_weight, eval_metric="logloss",
                n_jobs=-1, random_state=RANDOM_STATE,
            ),
        ),
    }

make_classifier_models(pd.Series([0, 0, 1])).keys()

dict_keys(['Logistic Regression', 'Random Forest', 'HistGradientBoosting', 'XGBoost'])

## Train and test the existing classification models

In [19]:
classification_rows = []
classification_data = {}

for dataset_name, frame in datasets.items():
    sample = frame[frame["classification_eligible"].eq(1)].copy()
    X = pd.get_dummies(
        sample[CLASSIFICATION_FEATURES],
        columns=CLASSIFICATION_CATEGORICAL,
        dummy_na=True,
        dtype=float,
    )
    y = sample["target_is_special_next_week"].astype(int)
    train, validation, test = split_masks(sample)

    for model_name, model in make_classifier_models(y.loc[train]).items():
        model.fit(X.loc[train], y.loc[train])
        validation_probability = model.predict_proba(X.loc[validation])[:, 1]
        threshold = best_f1_threshold(y.loc[validation], validation_probability)
        test_probability = model.predict_proba(X.loc[test])[:, 1]

        row = {"Dataset": dataset_name, "Model": model_name}
        row.update(classification_scores(y.loc[test], test_probability, threshold))
        row["Validation PR-AUC"] = average_precision_score(y.loc[validation], validation_probability)
        classification_rows.append(row)

    classification_data[dataset_name] = {
        "sample": sample, "X": X, "y": y,
        "train": train, "validation": validation, "test": test,
    }

classification_results = pd.DataFrame(classification_rows).sort_values(
    ["Dataset", "Validation PR-AUC"], ascending=[True, False]
).reset_index(drop=True)
classification_results.round(4)

,Dataset,Model,Threshold,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC,Validation PR-AUC
0,2024 + 2025 VIC,XGBoost,0.6625,0.7870,0.6361,0.1996,0.4520,0.2769,0.7006,0.2086,0.3152
1,2024 + 2025 VIC,HistGradientBoosting,0.6659,0.7916,0.6453,0.2081,0.4669,0.2879,0.7002,0.1996,0.3072
2,2024 + 2025 VIC,Random Forest,0.5051,0.8160,0.5819,0.1815,0.2964,0.2252,0.6633,0.1668,0.2932
3,2024 + 2025 VIC,Logistic Regression,0.7482,0.7967,0.6153,0.1930,0.3940,0.2591,0.6789,0.1871,0.2910
4,2025 only,HistGradientBoosting,0.6609,0.7950,0.5988,0.1805,0.3593,0.2403,0.6604,0.1825,0.3163
5,2025 only,XGBoost,0.6838,0.8370,0.5793,0.1983,0.2649,0.2268,0.6594,0.1816,0.3134
6,2025 only,Logistic Regression,0.6880,0.7702,0.6284,0.1853,0.4553,0.2634,0.6701,0.1853,0.2939
7,2025 only,Random Forest,0.4601,0.8003,0.5778,0.1677,0.3063,0.2168,0.6385,0.1609,0.2738


## Initialize the existing discount regression models

In [20]:
def make_regression_models():
    return {
        "Median Baseline": make_pipeline(
            SimpleImputer(strategy="median"), DummyRegressor(strategy="median")
        ),
        "Ridge Regression": make_pipeline(
            SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=10)
        ),
        "Random Forest": make_pipeline(
            SimpleImputer(strategy="median"),
            RandomForestRegressor(
                n_estimators=300, max_depth=14, min_samples_leaf=2,
                n_jobs=-1, random_state=RANDOM_STATE,
            ),
        ),
        "Extra Trees": make_pipeline(
            SimpleImputer(strategy="median"),
            ExtraTreesRegressor(
                n_estimators=300, max_depth=14, min_samples_leaf=2,
                n_jobs=-1, random_state=RANDOM_STATE,
            ),
        ),
        "HistGradientBoosting": make_pipeline(
            SimpleImputer(strategy="median"),
            HistGradientBoostingRegressor(
                max_iter=250, learning_rate=0.05, max_leaf_nodes=31,
                l2_regularization=1, random_state=RANDOM_STATE,
            ),
        ),
        "XGBoost": make_pipeline(
            SimpleImputer(strategy="median"),
            XGBRegressor(
                n_estimators=350, max_depth=4, learning_rate=0.04,
                subsample=0.8, colsample_bytree=0.8,
                objective="reg:squarederror", n_jobs=-1, random_state=RANDOM_STATE,
            ),
        ),
        "Neural Network (MLP)": make_pipeline(
            SimpleImputer(strategy="median"), StandardScaler(),
            MLPRegressor(
                hidden_layer_sizes=(64, 32), early_stopping=True,
                max_iter=600, random_state=RANDOM_STATE,
            ),
        ),
    }

make_regression_models().keys()

dict_keys(['Median Baseline', 'Ridge Regression', 'Random Forest', 'Extra Trees', 'HistGradientBoosting', 'XGBoost', 'Neural Network (MLP)'])

## Train and test the existing discount regression models

In [21]:
regression_rows = []
regression_data = {}

for dataset_name, frame in datasets.items():
    sample = frame[frame["discount_regression_eligible"].eq(1)].copy()
    X = pd.get_dummies(
        sample[REGRESSION_FEATURES],
        columns=REGRESSION_CATEGORICAL,
        dummy_na=True,
        dtype=float,
    )
    y = sample["target_discount_percent_next_week"].astype(float)
    train, validation, test = split_masks(sample)

    for model_name, model in make_regression_models().items():
        model.fit(X.loc[train], y.loc[train])
        validation_prediction = np.clip(model.predict(X.loc[validation]), 0, 80)
        test_prediction = np.clip(model.predict(X.loc[test]), 0, 80)
        regression_rows.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "Validation MAE": mean_absolute_error(y.loc[validation], validation_prediction),
            "Test MAE": mean_absolute_error(y.loc[test], test_prediction),
            "Test RMSE": mean_squared_error(y.loc[test], test_prediction) ** 0.5,
            "Test R2": r2_score(y.loc[test], test_prediction),
        })

    regression_data[dataset_name] = {
        "sample": sample, "X": X, "y": y,
        "train": train, "validation": validation, "test": test,
    }

regression_results = pd.DataFrame(regression_rows).sort_values(
    ["Dataset", "Validation MAE"]
).reset_index(drop=True)
regression_results.round(3)

,Dataset,Model,Validation MAE,Test MAE,Test RMSE,Test R2
0,2024 + 2025 VIC,HistGradientBoosting,7.604,8.462,10.771,0.403
1,2024 + 2025 VIC,Random Forest,7.719,8.732,10.741,0.406
2,2024 + 2025 VIC,XGBoost,8.046,9.161,11.064,0.370
3,2024 + 2025 VIC,Extra Trees,8.205,9.289,11.336,0.339
4,2024 + 2025 VIC,Neural Network (MLP),10.282,11.021,13.351,0.083
5,2024 + 2025 VIC,Ridge Regression,10.303,11.183,13.435,0.071
6,2024 + 2025 VIC,Median Baseline,11.626,12.027,14.069,-0.019
7,2025 only,HistGradientBoosting,8.015,9.139,11.267,0.347
8,2025 only,Random Forest,8.264,9.391,11.291,0.344
9,2025 only,Extra Trees,8.308,9.716,11.764,0.288


## Prepare product-level LSTM sequences

In [22]:
def build_sequences(frame, transformed_rows, target_column, eligibility_column):
    sequences, targets, dates = [], [], []
    frame = frame.reset_index(drop=True)
    transformed_rows = np.asarray(transformed_rows)

    groups = frame.groupby(["retailer", "region", "product_id"], sort=False).indices.values()
    for indexes in groups:
        indexes = np.asarray(indexes)
        weeks = frame.loc[indexes, "week_index"].to_numpy()
        for end in range(SEQUENCE_LENGTH - 1, len(indexes)):
            start = end - SEQUENCE_LENGTH + 1
            window = indexes[start:end + 1]
            if np.any(np.diff(weeks[start:end + 1]) != 1):
                continue
            last = indexes[end]
            if frame.at[last, eligibility_column] != 1 or pd.isna(frame.at[last, target_column]):
                continue
            sequences.append(transformed_rows[window])
            targets.append(frame.at[last, target_column])
            dates.append(frame.at[last, "catalogue_start_date"])

    return np.asarray(sequences, dtype="float32"), np.asarray(targets), pd.to_datetime(dates)

def prepare_lstm_data(frame, target_column, eligibility_column):
    ordered = frame.sort_values(
        ["retailer", "region", "product_id", "catalogue_start_date"]
    ).reset_index(drop=True)
    training_rows = ordered["catalogue_start_date"].lt(validation_start)

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    imputer.fit(ordered.loc[training_rows, LSTM_FEATURES])
    scaler.fit(imputer.transform(ordered.loc[training_rows, LSTM_FEATURES]))
    transformed = scaler.transform(imputer.transform(ordered[LSTM_FEATURES]))

    X, y, dates = build_sequences(
        ordered, transformed, target_column, eligibility_column
    )
    masks = (
        dates < validation_start,
        (dates >= validation_start) & (dates < test_start),
        dates >= test_start,
    )
    return X, y, masks, imputer, scaler

## Train and test LSTM models

In [23]:
import tensorflow as tf
from tensorflow import keras

tf.keras.utils.set_random_seed(RANDOM_STATE)
lstm_classification_rows = []
lstm_regression_rows = []

for dataset_name, frame in datasets.items():
    print(f"Training LSTM models: {dataset_name}")

    Xc, yc, classification_masks, classifier_imputer, classifier_scaler = prepare_lstm_data(
        frame, "target_is_special_next_week", "classification_eligible"
    )
    train_c, validation_c, test_c = classification_masks
    classifier = keras.Sequential([
        keras.layers.Input((SEQUENCE_LENGTH, Xc.shape[2])),
        keras.layers.LSTM(32),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    classifier.compile(optimizer="adam", loss="binary_crossentropy")
    positive_count = yc[train_c].sum()
    class_weight = {0: 1.0, 1: float((train_c.sum() - positive_count) / positive_count)}
    classifier.fit(
        Xc[train_c], yc[train_c],
        validation_data=(Xc[validation_c], yc[validation_c]),
        epochs=20, batch_size=256, class_weight=class_weight,
        callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
        verbose=0,
    )
    validation_probability = classifier.predict(Xc[validation_c], verbose=0).ravel()
    threshold = best_f1_threshold(yc[validation_c], validation_probability)
    test_probability = classifier.predict(Xc[test_c], verbose=0).ravel()
    row = {"Dataset": dataset_name, "Model": "LSTM"}
    row.update(classification_scores(yc[test_c], test_probability, threshold))
    row["Validation PR-AUC"] = average_precision_score(yc[validation_c], validation_probability)
    lstm_classification_rows.append(row)

    Xr, yr, regression_masks, regressor_imputer, regressor_scaler = prepare_lstm_data(
        frame, "target_discount_percent_next_week", "discount_regression_eligible"
    )
    train_r, validation_r, test_r = regression_masks
    regressor = keras.Sequential([
        keras.layers.Input((SEQUENCE_LENGTH, Xr.shape[2])),
        keras.layers.LSTM(32),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1),
    ])
    regressor.compile(optimizer="adam", loss="mae")
    regressor.fit(
        Xr[train_r], yr[train_r],
        validation_data=(Xr[validation_r], yr[validation_r]),
        epochs=25, batch_size=128,
        callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
        verbose=0,
    )
    validation_prediction = np.clip(
        regressor.predict(Xr[validation_r], verbose=0).ravel(), 0, 80
    )
    test_prediction = np.clip(
        regressor.predict(Xr[test_r], verbose=0).ravel(), 0, 80
    )
    lstm_regression_rows.append({
        "Dataset": dataset_name,
        "Model": "LSTM",
        "Validation MAE": mean_absolute_error(yr[validation_r], validation_prediction),
        "Test MAE": mean_absolute_error(yr[test_r], test_prediction),
        "Test RMSE": mean_squared_error(yr[test_r], test_prediction) ** 0.5,
        "Test R2": r2_score(yr[test_r], test_prediction),
    })

    if dataset_name == "2024 + 2025 VIC":
        classifier.save(DATA_DIR / "discount_special_lstm_combined.keras")
        regressor.save(DATA_DIR / "discount_price_lstm_combined.keras")
        joblib.dump({
            "features": LSTM_FEATURES,
            "sequence_length": SEQUENCE_LENGTH,
            "classifier_threshold": threshold,
            "classifier_imputer": classifier_imputer,
            "classifier_scaler": classifier_scaler,
            "regressor_imputer": regressor_imputer,
            "regressor_scaler": regressor_scaler,
        }, DATA_DIR / "discount_lstm_preprocessing_combined.joblib")

Training LSTM models: 2025 only
Training LSTM models: 2024 + 2025 VIC


## Compare all models

In [24]:
classification_results = pd.concat(
    [classification_results, pd.DataFrame(lstm_classification_rows)], ignore_index=True
).sort_values(["Dataset", "Validation PR-AUC"], ascending=[True, False]).reset_index(drop=True)

regression_results = pd.concat(
    [regression_results, pd.DataFrame(lstm_regression_rows)], ignore_index=True
).sort_values(["Dataset", "Validation MAE"]).reset_index(drop=True)

classification_results.to_csv(
    DATA_DIR / "classification_2025_vs_combined_results.csv", index=False
)
regression_results.to_csv(
    DATA_DIR / "regression_2025_vs_combined_results.csv", index=False
)

display(classification_results[
    ["Dataset", "Model", "Accuracy", "Precision", "Recall", "F1", "PR-AUC", "Validation PR-AUC"]
].round(4))

display(regression_results[
    ["Dataset", "Model", "Validation MAE", "Test MAE", "Test RMSE", "Test R2"]
].round(3))

,Dataset,Model,Accuracy,Precision,Recall,F1,PR-AUC,Validation PR-AUC
0,2024 + 2025 VIC,LSTM,0.7713,0.2016,0.4628,0.2809,0.2216,0.3489
1,2024 + 2025 VIC,XGBoost,0.7870,0.1996,0.4520,0.2769,0.2086,0.3152
2,2024 + 2025 VIC,HistGradientBoosting,0.7916,0.2081,0.4669,0.2879,0.1996,0.3072
3,2024 + 2025 VIC,Random Forest,0.8160,0.1815,0.2964,0.2252,0.1668,0.2932
4,2024 + 2025 VIC,Logistic Regression,0.7967,0.1930,0.3940,0.2591,0.1871,0.2910
5,2025 only,LSTM,0.8485,0.2432,0.2698,0.2558,0.2167,0.3544
6,2025 only,HistGradientBoosting,0.7950,0.1805,0.3593,0.2403,0.1825,0.3163
7,2025 only,XGBoost,0.8370,0.1983,0.2649,0.2268,0.1816,0.3134
8,2025 only,Logistic Regression,0.7702,0.1853,0.4553,0.2634,0.1853,0.2939
9,2025 only,Random Forest,0.8003,0.1677,0.3063,0.2168,0.1609,0.2738


,Dataset,Model,Validation MAE,Test MAE,Test RMSE,Test R2
0,2024 + 2025 VIC,HistGradientBoosting,7.604,8.462,10.771,0.403
1,2024 + 2025 VIC,Random Forest,7.719,8.732,10.741,0.406
2,2024 + 2025 VIC,XGBoost,8.046,9.161,11.064,0.370
3,2024 + 2025 VIC,LSTM,8.107,10.280,12.493,0.128
4,2024 + 2025 VIC,Extra Trees,8.205,9.289,11.336,0.339
5,2024 + 2025 VIC,Neural Network (MLP),10.282,11.021,13.351,0.083
6,2024 + 2025 VIC,Ridge Regression,10.303,11.183,13.435,0.071
7,2024 + 2025 VIC,Median Baseline,11.626,12.027,14.069,-0.019
8,2025 only,HistGradientBoosting,8.015,9.139,11.267,0.347
9,2025 only,Random Forest,8.264,9.391,11.291,0.344


## Save the best combined tabular model

In [25]:
combined_name = "2024 + 2025 VIC"
combined_classification = classification_results[
    (classification_results["Dataset"] == combined_name)
    & (classification_results["Model"] != "LSTM")
]
best_classifier_row = combined_classification.iloc[0]

combined_regression = regression_results[
    (regression_results["Dataset"] == combined_name)
    & (regression_results["Model"] != "LSTM")
]
best_regressor_row = combined_regression.iloc[0]

classifier_values = classification_data[combined_name]
regression_values = regression_data[combined_name]
final_classifier = clone(
    make_classifier_models(classifier_values["y"])[best_classifier_row["Model"]]
).fit(classifier_values["X"], classifier_values["y"])
final_regressor = clone(
    make_regression_models()[best_regressor_row["Model"]]
).fit(regression_values["X"], regression_values["y"])

model_bundle = {
    "classifier": final_classifier,
    "classifier_name": best_classifier_row["Model"],
    "classifier_threshold": float(best_classifier_row["Threshold"]),
    "classifier_raw_features": CLASSIFICATION_FEATURES,
    "classifier_columns": classifier_values["X"].columns.tolist(),
    "regressor": final_regressor,
    "regressor_name": best_regressor_row["Model"],
    "regressor_raw_features": REGRESSION_FEATURES,
    "regressor_columns": regression_values["X"].columns.tolist(),
    "categorical_features": REGRESSION_CATEGORICAL,
    "discount_rounding_step": 5,
}
joblib.dump(model_bundle, DATA_DIR / "discount_price_prediction_model_combined.joblib")

print(f"Saved classifier: {best_classifier_row['Model']}")
print(f"Saved regressor: {best_regressor_row['Model']}")

Saved classifier: XGBoost
Saved regressor: HistGradientBoosting


## Test the saved model on three random products

In [26]:
MODEL_FILE = DATA_DIR / "discount_price_prediction_model_combined.joblib"
saved_model = joblib.load(MODEL_FILE)

def prepare_saved_model_input(rows, raw_features, model_columns, categorical_features):
    values = pd.get_dummies(
        rows[raw_features],
        columns=categorical_features,
        dummy_na=True,
        dtype=float,
    )
    return values.reindex(columns=model_columns, fill_value=0)

def predict_next_week_offer(rows, bundle):
    classifier_input = prepare_saved_model_input(
        rows,
        bundle["classifier_raw_features"],
        bundle["classifier_columns"],
        CLASSIFICATION_CATEGORICAL,
    )
    regressor_input = prepare_saved_model_input(
        rows,
        bundle["regressor_raw_features"],
        bundle["regressor_columns"],
        bundle["categorical_features"],
    )

    special_probability = bundle["classifier"].predict_proba(classifier_input)[:, 1]
    raw_discount = bundle["regressor"].predict(regressor_input)
    rounding_step = bundle.get("discount_rounding_step", 5)
    predicted_discount = np.clip(
        np.round(raw_discount / rounding_step) * rounding_step, 0, 80
    )
    regular_price = pd.to_numeric(rows["regular_price_filled"], errors="coerce").to_numpy()
    predicted_special = special_probability >= bundle["classifier_threshold"]
    predicted_special_price = regular_price * (1 - predicted_discount / 100)
    expected_price = regular_price * (1 - special_probability * predicted_discount / 100)

    output = rows[
        ["product_id", "original_product_name", "catalogue_start_date"]
    ].reset_index(drop=True).copy()
    output["regular_price"] = np.round(regular_price, 2)
    output["special_probability"] = np.round(special_probability, 3)
    output["predicted_is_special"] = predicted_special
    output["predicted_discount_if_special"] = predicted_discount.astype(int)
    output["predicted_special_price"] = np.round(predicted_special_price, 2)
    output["probability_weighted_expected_price"] = np.round(expected_price, 2)
    return output

latest_products = (
    datasets[combined_name]
    .sort_values("catalogue_start_date")
    .groupby(["retailer", "region", "product_id"], as_index=False)
    .tail(1)
)
eligible_products = latest_products[
    latest_products["regular_price_filled"].notna()
].copy()

random_products = eligible_products.sample(n=3, random_state=RANDOM_STATE)
random_predictions = predict_next_week_offer(random_products, saved_model)

print(f"Loaded: {MODEL_FILE.name}")
print(f"Classifier: {saved_model['classifier_name']}")
print(f"Discount regressor: {saved_model['regressor_name']}")
display(random_predictions)

Loaded: discount_price_prediction_model_combined.joblib
Classifier: XGBoost
Discount regressor: HistGradientBoosting


,product_id,original_product_name,catalogue_start_date,regular_price,special_probability,predicted_is_special,predicted_discount_if_special,predicted_special_price,probability_weighted_expected_price
0,P00629,Sunbeam Slivered Almonds 100g,2025-02-05,5.5,0.144,False,45,3.03,5.14
1,P08437,Dove Advanced 72hr Roll On Deodorant 50mL,2025-12-31,6.0,0.266,False,50,3.00,5.20
2,P05208,Bic 4 Colour Pastel Pens 3 Pack,2025-07-02,10.0,0.127,False,45,5.50,9.43


## Best validation-selected results

In [27]:
best_classifiers = classification_results.loc[
    classification_results.groupby("Dataset")["Validation PR-AUC"].idxmax(),
    ["Dataset", "Model", "Accuracy", "Precision", "Recall", "F1", "PR-AUC"],
]
best_regressors = regression_results.loc[
    regression_results.groupby("Dataset")["Validation MAE"].idxmin(),
    ["Dataset", "Model", "Validation MAE", "Test MAE", "Test RMSE", "Test R2"],
]

display(best_classifiers.round(4))
display(best_regressors.round(3))

,Dataset,Model,Accuracy,Precision,Recall,F1,PR-AUC
0,2024 + 2025 VIC,LSTM,0.7713,0.2016,0.4628,0.2809,0.2216
5,2025 only,LSTM,0.8485,0.2432,0.2698,0.2558,0.2167


,Dataset,Model,Validation MAE,Test MAE,Test RMSE,Test R2
0,2024 + 2025 VIC,HistGradientBoosting,7.604,8.462,10.771,0.403
8,2025 only,HistGradientBoosting,8.015,9.139,11.267,0.347
